In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="NightPrince/MasriSpeech-Full", 
    repo_type="dataset", local_dir="./MasriSpeech-Full", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 24 files: 100%|██████████| 24/24 [00:10<00:00,  2.32it/s]


'/home/ubuntu/MasriSpeech-Full'

In [14]:
files = glob('MasriSpeech-Full/*/*.parquet')
len(files)

24

In [15]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            try:
                t = df['transcription'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': t,
                    'speaker': f"{base}"
                })
            except Exception as e:
                pass
        
    return data

In [16]:
# data = loop((files[:1], 0))

In [ ]:
data = multiprocessing(files, loop, cores = len(files))

  7%|▋         | 145/2205 [00:18<03:34,  9.59it/s]